# 03 - Markov Chain (order-1 & order-2)

The first model with actual memory: score *"what usually follows city `l` (order-1), or the pair `(l2, l1)` (order-2)?"* by counting observed transitions in the training trips.

$$\text{score}(i \mid \text{context}) = \text{count}(\text{context} \to i) + k \quad \text{(add-}k\text{ smoothing)}$$

No learning, no gradient -- just counts. This is the first rung above Popularity on the sophistication ladder (Popularity -> **Markov** -> FPMC -> GRU4Rec -> SASRec), and its ceiling is exactly what it can't do: generalise beyond a context it has literally seen before.

All logic is in `src/models/markov.py`; this notebook grid-searches order and smoothing (selected by validation NDCG@10, same protocol as every other model), then runs the selected configuration and logs it through the shared `evaluate.py`.

In [ ]:
import os, sys
def find_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'src', 'preprocessing.py')):
            return d
        d = os.path.dirname(d)
    return os.path.abspath(os.path.join(os.getcwd(), '..'))
ROOT = find_root(); sys.path.insert(0, ROOT)
from src.preprocessing import load_processed, preprocess
try:
    data = load_processed(os.path.join(ROOT, 'data', 'processed'))
except FileNotFoundError:
    data = preprocess(data_dir=os.path.join(ROOT, 'data'), sample_n_trips=5000)
print('loaded:', data['meta'])

## Ablation -- order & smoothing, selected by validation NDCG@10

Markov's only real hyperparameters are the context length (order 1 vs 2) and the add-$k$ (Laplace) smoothing constant. Exactly like the neural models' grids, we pick both by validation NDCG@10 and never touch the test set during selection -- `select_markov` does this grid search for us.

In [ ]:
from src.models.markov import select_markov, run_markov
from src.evaluate import append_result
RESULTS = os.path.join(ROOT, 'results.csv')
import pandas as pd

results = {}
for order in (1, 2):
    for smoothing in (0.1, 1.0, 5.0):
        label = f"Markov-order{order}-k{smoothing}"
        metrics, ranked, val_ndcg = run_markov(
            data, order=order, smoothing=smoothing, verbose=False
        )
        metrics["val_NDCG@10"] = round(val_ndcg, 4)
        results[label] = metrics
        append_result(RESULTS, label, metrics)   # logs every combo to results.csv
        print(f"--- {label} ---")
        print("TEST:", {k: round(v, 4) if isinstance(v, float) else v
                         for k, v in metrics.items()})
        print()

## The fallback rate -- why Markov plateaus

Whenever a test trip's exact context (its last 1 or 2 cities) was never seen in training, the model has no real answer and falls back to a lower order -- or, in the worst case, to plain Popularity. `metrics['fallback_rate']` is the fraction of test trips that needed this fallback. It is the single number that explains Markov's ceiling: no amount of smoothing fixes a context the model has never observed, which is exactly the gap FPMC's factorization and GRU4Rec/SASRec's learned embeddings are built to close.

In [ ]:
print(f"fallback rate on test: {metrics['fallback_rate']:.1%}")

# also report order-1 and order-2 individually (fixed smoothing) for the
# report's comparison table, beyond just the selected config
for order in (1, 2):
    m, _, val = run_markov(data, order=order, smoothing=best_cfg['smoothing'], verbose=False)
    print(f"order={order}: val_NDCG@10={val:.4f} test_NDCG@10={m['NDCG@10']:.4f} "
          f"fallback_rate={m['fallback_rate']:.1%}")

## What to write up

- Markov should comfortably beat Popularity wherever the immediate last-city transition carries signal, but plateau earlier than FPMC/GRU4Rec/SASRec on longer trips (see `06_comparison.ipynb`'s trip-length slicing).
- Report the selected `(order, smoothing)` and the **fallback rate** together -- a high fallback rate is direct evidence for *why* a fixed-order count table can't generalise the way a learned model can.
- `n_params` here is the number of distinct `(context, next-city)` counts stored, the count-based analogue of a neural model's parameter count, useful for the accuracy-vs-cost discussion in `06`.